# Step 8 — BAGEL first-divergent-operator localization

This is a bounded **correctness diagnostic**, not a performance benchmark. It keeps BAGEL's original `num_timesteps=50` and therefore preserves the exact timestep and Euler-step-size math, but stops the graph after only **8 refinement iterations**. We do not need 49 iterations to locate a discrepancy that Experiment 7 showed within the first two consecutive B=2 executions.

The diagnostic runs four isolated arms: serial B=1; B=2 with partner seed A; an exact repeat of that B=2 arm; and B=2 with only the partner seed changed. The fixed target is always request 0 with seed `20261016`. The repeat distinguishes partner sensitivity from run-to-run nondeterminism.

Post-step latents are saved for steps 0–7. Internal stages are fingerprinted for steps 0–3, which covers the expected first B=2 boundary while keeping artifacts small. Large tensors use exact SHA-256 fingerprints of all tensor bytes plus deterministic numerical samples; small tensors are also retained in full. The analyzer checks timestep metadata, packing indexes, VAE projection, timestep/position embeddings, the embedding add, packed transformer input, transformer hidden states, LLM-to-VAE velocity projection, CFG combination, Euler update, output splitting, and state handoff.

Use an **A100 High-RAM** runtime and run top to bottom. Allow roughly **35–60 minutes after dependency/checkpoint setup** for four fresh server launches, though the actual model work is far shorter than Step 7. The long cell emits heartbeats, owns and cleans up every server, and packages partial evidence even if analysis fails.

## 1. Mount Drive and create local-only active paths

Drive is used only for the final verified ZIP. Active logs, traces, fingerprints, images, and the checkpoint cache remain under `/content`.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

drive.mount('/content/drive')
RUN_ID = 'operator_localization_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = '9a2abb613bb6553065cb483f88b077e28d08210d'
REPO = Path('/content') / f'mstar_{RUN_ID}'
LOCAL_ROOT = Path('/content/bagel_runs')
RESULT_DIR = LOCAL_ROOT / RUN_ID
DIAGNOSTIC_DIR = RESULT_DIR / 'first_divergent_operator'
CACHE_DIR = Path('/content/hf_cache_bagel')
UV_CACHE_DIR = Path('/content/uv_cache')
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL')
for path in (LOCAL_ROOT, RESULT_DIR, CACHE_DIR, UV_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=path != RESULT_DIR)
os.environ['UV_CACHE_DIR'] = str(UV_CACHE_DIR)
print({'run_id': RUN_ID, 'active_results': str(RESULT_DIR),
       'drive_backup': str(DRIVE_BACKUP_DIR)})

## 2. Clone the exact implementation and verify hardware

This stops before dependency or checkpoint downloads if the runtime lacks an 80 GB-class GPU, sufficient host RAM, or sufficient local disk.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content', '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50', '--require-disk-free-gib', '55',
], check=True)
source = {
    'fork_url': FORK_URL, 'requested_commit': RUN_COMMIT,
    'resolved_commit': head,
    'git_status': subprocess.check_output(
        ['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
print('Hardware preflight passed at', head)

## 3. Create or resume the pinned virtual environment

This is re-runnable within the same Colab VM. All CPU and real-PyTorch diagnostic tests run before any model server starts.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output(
    [sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'],
    text=True).strip()
VENV = REPO / '.venv-bagel'
VENV_PY = VENV / 'bin/python'
if not VENV_PY.exists():
    subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]',
    'matplotlib==3.10.7',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PATH'] = os.pathsep.join((str(VENV / 'bin'), env['PATH']))
tests = (
    'test_bagel_image_batch_packing.py', 'test_execution_trace.py',
    'test_bagel_experiment_runtime.py', 'test_bagel_pruned_batch_gate.py',
    'test_bagel_operator_localization.py',
)
for test in tests:
    subprocess.run([str(VENV_PY), str(REPO / 'test/modular' / test), '-v'],
                   cwd=REPO, env=env, check=True)
gpu_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import torch; assert torch.cuda.is_available(); '
    'x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); '
    'y=x@x; torch.cuda.synchronize(); '
    'print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), float(y[0,0]))',
], text=True, env=env)
(RESULT_DIR / 'gpu_import_check.txt').write_text(gpu_check)
print('Environment preflight passed:', gpu_check.strip())

## 4. Reuse or download the pinned BAGEL checkpoint locally

A second run in the same VM reuses `/content/hf_cache_bagel`. Every safetensors header is validated before a server starts.

In [ ]:
manifest_path = RESULT_DIR / 'checkpoint_manifest.json'
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR), '--manifest', str(manifest_path),
], cwd=REPO, env=env, check=True)
manifest = json.loads(manifest_path.read_text())
print('Validated checkpoint:', manifest['resolved_revision'])

## 5. Run the complete four-arm diagnostic

Run this cell once and leave it running while heartbeats continue. Each arm gets a fresh owned server. On failure or interruption, all evidence collected so far is packaged and the owned process group is stopped. Do not rerun this cell with the same `RUN_ID`.

In [ ]:
sys.path.insert(0, str(REPO / 'experiments/bagel/scripts'))
from experiment_runtime import package_results, run_logged
command = [
    str(VENV_PY),
    str(REPO / 'experiments/bagel/scripts/run_operator_localization.py'),
    '--repo-root', str(REPO), '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(DIAGNOSTIC_DIR), '--port', '8031',
]
run_error = None
try:
    run_logged(command, RESULT_DIR / 'diagnostic_console.log', cwd=REPO, env=env,
               stage='complete Step 8 operator-localization diagnostic',
               timeout=14400)
except BaseException as exc:
    run_error = exc
finally:
    archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
    if archive_path.exists():
        print('Preserving existing local ZIP:', archive_path)
    else:
        receipt = package_results(RESULT_DIR, DRIVE_BACKUP_DIR)
        receipt_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_backup_receipt.json'
        receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
        print(json.dumps(receipt, indent=2))
if run_error is not None:
    raise run_error
print('Diagnostic completed:', DIAGNOSTIC_DIR)

## 6. Inspect the causal result and download the verified ZIP

The most useful result is `first_nonexact_stage_at_boundary`. For example, `hidden_main` points to the transformer forward under the changed packed shape; `velocity_main` points later to the LLM-to-VAE projection; `combined_velocity` points to CFG combination; and `output_latent_pre_split` points to the Euler update. The report only makes that classification when the incoming target latent and all control tensors are exact at the boundary, the boundary actually ran at B=2, the result repeats, and the partner-control schedules match.

In [ ]:
report = json.loads((DIAGNOSTIC_DIR / 'operator_localization.json').read_text())
localized = {}
for arm, comparison in report['serial_vs_batched'].items():
    first_step = comparison['first_nonexact_output_step']
    boundary = next((row for row in comparison['steps']
                     if row['step'] == first_step), None)
    localized[arm] = {
        'first_nonexact_output_step': first_step,
        'first_nonexact_output_was_b2': comparison['first_nonexact_output_was_b2'],
        'input_exact_at_boundary': comparison['input_exact_at_first_nonexact_output'],
        'controls_exact_at_boundary': comparison['controls_exact_at_first_nonexact_output'],
        'first_nonexact_stage': comparison['first_nonexact_stage_at_boundary'],
        'changed_stages_at_boundary': ([] if boundary is None else [
            name for name, metrics in boundary['stages'].items()
            if not metrics['exact']
        ]),
    }
compact = {
    'status': report['status'],
    'performance_authorized': report['performance_authorized'],
    'hypothesis_assessment': report['hypothesis_assessment'],
    'configured_num_timesteps': report['configured_num_timesteps'],
    'executed_refinement_steps': report['executed_refinement_steps'],
    'b2_target_steps': report['b2_target_steps'],
    'localized_boundaries': localized,
    'b2_repeatability': {
        key: report['b2_repeatability'][key] for key in (
            'partner_seeds', 'partner_inputs_differ',
            'target_schedule_equal', 'all_target_stages_exact')
    },
    'partner_isolation': {
        key: report['partner_isolation'][key] for key in (
            'partner_seeds', 'partner_inputs_differ',
            'target_schedule_equal', 'all_target_stages_exact')
    },
}
print(json.dumps(compact, indent=2))
archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Verified ZIP:', archive_path, 'SHA-256:', digest)
from google.colab import files
files.download(str(archive_path))

After the browser download starts, share the ZIP and the printed compact report. Then use **Runtime → Disconnect and delete runtime**. Do not use timing from this notebook for a performance claim: tensor capture deliberately synchronizes the GPU.